# News Forecast Pipeline — 5 крупнейших СМИ РФ

**Цель:** Сбор → ETL → Анализ → Бэктест → Прогноз на **02.04.2026**

**СМИ:** Коммерсантъ, Коммерсантъ, Лента.ру, Интерфакс

| Шаг | Ячейка | Описание |
|-----|--------|----------|
| 0 | Setup | Настройка окружения и импорты |
| 1 | Scrape | Сбор данных за 90 дней (RSS + архив) |
| 2 | ETL | Очистка, дедупликация, фильтрация |
| 3 | Analyze | Топики, частоты, NER, noise check |
| 4 | Backtest | Holdout-проверка на последних 7 днях |
| 5 | Metrics | Оценка качества прогноза |
| 6 | Forecast | Прогноз на целевую дату |
| 7 | Results | Просмотр итогового JSON |

---
## Ячейка 0 — Setup: окружение и импорты

In [3]:
import sys, os

# Убедимся, что titles/ на sys.path
TITLES_DIR = os.path.dirname(os.path.abspath("__file__"))
if TITLES_DIR not in sys.path:
    sys.path.insert(0, TITLES_DIR)

# Проверяем версию Python
print(f"Python: {sys.version}")
print(f"Working dir: {os.getcwd()}")

Python: 3.14.3 (main, Mar  3 2026, 15:00:44) [MSC v.1944 64 bit (AMD64)]
Working dir: c:\Users\xaxhd\OneDrive\Рабочий стол\TEst\titles


In [4]:
# Проверка наличия всех зависимостей
missing = []
for pkg in ["feedparser", "requests", "bs4", "lxml", "pandas", "numpy",
            "sklearn", "tqdm", "nltk", "plotly", "razdel", "openai", "python-dotenv", "pymystem3"]:
    try:
        __import__(pkg)
    except ImportError:
        missing.append(pkg)

if missing:
    print(f"⚠️  Отсутствуют пакеты: {missing}")
    print("Запустите: uv uv pip install -r requirements.txt")
else:
    print("✅ Все зависимости установлены")

⚠️  Отсутствуют пакеты: ['feedparser', 'requests', 'bs4', 'lxml', 'pandas', 'numpy', 'sklearn', 'tqdm', 'nltk', 'plotly', 'razdel', 'openai', 'pymystem3']
Запустите: pip install -r requirements.txt


In [5]:
uv uv pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


error: externally-managed-environment

× This environment is externally managed
╰─> This Python installation is managed by uv and should not be modified.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detailed specification.


In [6]:
# Загружаем конфиг и проверяем пути
import config
import datetime

print(f"История: {config.HISTORY_FROM}  →  {config.TODAY}  ({config.HISTORY_DAYS} дней)")
print(f"Целевая дата прогноза: {config.TARGET_DATE}")
print(f"\nСМИ:")
for slug, cfg in config.OUTLETS.items():
    leads = "✅ лид" if cfg["has_lead"] else "❌ только заголовок (paywall)"
    print(f"  {slug:<12} {cfg['name']:<20} {leads}")

print(f"\nДиректории:")
for d in [config.RAW_DIR, config.CLEAN_DIR, config.FORECASTS_DIR, config.EVENTS_DIR]:
    exists = "✅" if os.path.exists(d) else "❌ (создаётся автоматически)"
    print(f"  {exists}  {d}")

История: 2025-12-29  →  2026-03-29  (90 дней)
Целевая дата прогноза: 2026-04-02

СМИ:
  kommersant          Коммерсантъ                  ✅ лид
  kommersant   Коммерсантъ          ✅ лид
  deleted    Ведомости            ❌ только заголовок (paywall)
  lenta        Лента.ру             ✅ лид
  interfax     Интерфакс            ✅ лид

Директории:
  ✅  c:\Users\xaxhd\OneDrive\Рабочий стол\TEst\titles\data\raw
  ✅  c:\Users\xaxhd\OneDrive\Рабочий стол\TEst\titles\data\clean
  ✅  c:\Users\xaxhd\OneDrive\Рабочий стол\TEst\titles\data\forecasts
  ✅  c:\Users\xaxhd\OneDrive\Рабочий стол\TEst\titles\data\events


In [10]:
# Проверка OpenRouter (нужна перед шагом Forecast)
import config
if not getattr(config, 'OPENROUTER_API_KEY', None):
    print("⚠️  OPENROUTER_API_KEY не установлен в .env")
    print("   Получите ключ на https://openrouter.ai/ и добавьте в .env")
else:
    print(f"✅ OpenRouter сконфигурирован. Модель: {config.OPENROUTER_MODEL}")

✅ OpenRouter доступна. Модели: ['llama3.1:latest']


---
## Ячейка 1 — Scrape: сбор данных

> ⏱️ **~5-20 минут** в зависимости от скорости соединения и количества доступных архивных страниц.
> 
> Можно ограничить список СМИ через `OUTLETS_TO_SCRAPE` или уменьшить `START_DATE`.
> 
> Добавьте `enrich_leads=True` для полного скрапинга лидов (медленнее, по 1-2 сек/статья).

In [ ]:
import config
# ── Настройки скрапинга ──────────────────────────────────────────
OUTLETS_TO_SCRAPE = config.OUTLET_SLUGS      # все 5, или например ["kommersant", "lenta"]
SCRAPE_FROM       = config.HISTORY_FROM       # дата начала (по умолчанию: TODAY − 90 дней)
SCRAPE_TO         = config.TODAY
ENRICH_LEADS      = False                    # True = доп. запросы за каждым лидом (медленно)
# ────────────────────────────────────────────────────────────────

from scraper import scrape_all

scrape_results = scrape_all(
    slugs=OUTLETS_TO_SCRAPE,
    start_date=SCRAPE_FROM,
    end_date=SCRAPE_TO,
    enrich_leads=ENRICH_LEADS,
)

print("\n── Итог скрапинга ──")
for slug, recs in scrape_results.items():
    print(f"  {slug:<12} {len(recs):>5} записей")



[scraper] Коммерсантъ (kommersant) 2025-12-29 → 2026-03-29
  [rss] fetching 5 feeds...
  [rss] Warning: could not parse https://rss.kommersant.ru/v10/main.rss
  [rss] Warning: could not parse https://rss.kommersant.ru/v10/economics.rss
  [rss] Warning: could not parse https://rss.kommersant.ru/v10/politics.rss
  [rss] Warning: could not parse https://rss.kommersant.ru/v10/business.rss
  [rss] Warning: could not parse https://rss.kommersant.ru/v10/finance.rss
  [rss] got 0 entries
  [archive] got 0 new entries from archive pages
  [wayback] using Wayback CDX API fallback...
  [wayback] added 0 entries from Wayback
  [done] total 0 records for kommersant

[scraper] Коммерсантъ (kommersant) 2025-12-29 → 2026-03-29
  [rss] fetching 3 feeds...
  [rss] got 337 entries
  [archive] got 0 new entries from archive pages
  [done] total 337 records for kommersant

[scraper] Ведомости (deleted) 2025-12-29 → 2026-03-29
  [rss] fetching 1 feeds...
  [rss] got 200 entries
  [wayback] using Wayback CD

---
## Ячейка 2 — ETL: очистка и дедупликация

In [14]:
import importlib, etl as _etl_mod
importlib.reload(_etl_mod)
from etl import run_etl, load_clean
import pandas as pd

run_etl(slugs=config.OUTLET_SLUGS)

# Сводка по чистым данным
print("\n── Итог ETL ──")
summary_rows = []
for slug in config.OUTLET_SLUGS:
    df = load_clean(slug)
    if df.empty:
        summary_rows.append({"outlet": slug, "records": 0, "with_lead": 0,
                              "date_min": None, "date_max": None})
        continue
    df["published_at"] = pd.to_datetime(df["published_at"], errors="coerce")
    summary_rows.append({
        "outlet":   slug,
        "records":  len(df),
        "with_lead": df["lead"].notna().sum(),
        "date_min": df["published_at"].min().date() if not df.empty else None,
        "date_max": df["published_at"].max().date() if not df.empty else None,
    })

pd.DataFrame(summary_rows)


  [etl] kommersant: 0 raw records
  [etl] exact URL dedup: 0 -> 0
  [etl] kommersant: 0 clean records
  [etl] saved c:\Users\xaxhd\OneDrive\Рабочий стол\TEst\titles\data\clean\kommersant_clean.csv
  [etl] kommersant: 337 raw records
  [etl] exact URL dedup: 337 -> 337


c:\Users\xaxhd\OneDrive\Рабочий стол\TEst\titles\etl.py:130: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  url_mask    = df["url"].str.contains(OPINION_URL_PATTERNS, na=False)
c:\Users\xaxhd\OneDrive\Рабочий стол\TEst\titles\etl.py:130: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  url_mask    = df["url"].str.contains(OPINION_URL_PATTERNS, na=False)


  [etl] near-dedup removed 4 rows (threshold=0.85)
  [etl] kommersant: 333 clean records
  [etl] saved c:\Users\xaxhd\OneDrive\Рабочий стол\TEst\titles\data\clean\kommersant_clean.csv
  [etl] deleted: 200 raw records
  [etl] exact URL dedup: 200 -> 200
  [etl] near-dedup removed 2 rows (threshold=0.85)
  [etl] deleted: 198 clean records
  [etl] saved c:\Users\xaxhd\OneDrive\Рабочий стол\TEst\titles\data\clean\deleted_clean.csv
  [etl] lenta: 3696 raw records


c:\Users\xaxhd\OneDrive\Рабочий стол\TEst\titles\etl.py:130: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  url_mask    = df["url"].str.contains(OPINION_URL_PATTERNS, na=False)


  [etl] exact URL dedup: 3696 -> 3696
  [etl] opinion filter removed 1 rows


c:\Users\xaxhd\OneDrive\Рабочий стол\TEst\titles\etl.py:130: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  url_mask    = df["url"].str.contains(OPINION_URL_PATTERNS, na=False)


  [etl] near-dedup removed 812 rows (threshold=0.85)
  [etl] lenta: 2883 clean records
  [etl] saved c:\Users\xaxhd\OneDrive\Рабочий стол\TEst\titles\data\clean\lenta_clean.csv
  [etl] interfax: 5978 raw records
  [etl] exact URL dedup: 5978 -> 5978
  [etl] opinion filter removed 1 rows


c:\Users\xaxhd\OneDrive\Рабочий стол\TEst\titles\etl.py:130: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  url_mask    = df["url"].str.contains(OPINION_URL_PATTERNS, na=False)


  [etl] near-dedup removed 605 rows (threshold=0.85)
  [etl] interfax: 5372 clean records
  [etl] saved c:\Users\xaxhd\OneDrive\Рабочий стол\TEst\titles\data\clean\interfax_clean.csv

── Итог ETL ──


,outlet,records,with_lead,date_min,date_max
0,rbc,0,0,None,None
1,kommersant,333,333,2026-03-26,2026-03-29
2,vedomosti,198,198,2026-03-27,2026-03-29
3,lenta,2883,2883,2025-12-29,2026-03-29
4,interfax,5372,5372,2025-12-29,2026-03-29


In [15]:
# Просмотр нескольких строк для одного СМИ
PREVIEW_OUTLET = "kommersant"   # поменяйте на любой slug

df_preview = load_clean(PREVIEW_OUTLET)
df_preview["published_at"] = pd.to_datetime(df_preview["published_at"], errors="coerce")
df_preview.sort_values("published_at", ascending=False)[["published_at", "rubric", "title", "lead"]].head(10)

,published_at,rubric,title,lead


---
## Ячейка 3 — Analyze: темы, частоты, NER, noise check

In [ ]:
import config
from analyzer import analyze_all

analysis = analyze_all(slugs=config.OUTLET_SLUGS)


  [analyze] No clean data for kommersant

[analyze] kommersant: 333 records
  [noise] kommersant: daily_mean=83.2, cv=0.928, stable=False


Installing mystem to C:\Users\xaxhd/.local/bin\mystem.exe from http://download.cdn.yandex.net/mystem/mystem-3.1-win-64bit.zip


  [topics] top-5:
    россии / делу / пресс служба / служба / пресс: 36 (10.8%)
    сша / ирана / агентство / reuters / трампа: 32 (9.6%)
    года / марта / вице / премьер / россии: 29 (8.7%)
    области / всу / telegram / канале / telegram канале: 24 (7.2%)
    мид / мира / сша / которых / связи: 22 (6.6%)
  [entities] top persons: {}
  [style]  avg_title=9.5 words, avg_lead=28.1 words

[analyze] deleted: 198 records
  [noise] deleted: daily_mean=66.0, cv=0.265, stable=True
  [topics] top-5:
    сша / иран / ираном / апреля / ирана: 140 (70.7%)
    путин / россии / россия / президентом / москва: 9 (4.5%)
    рф / ушаков / депутатами / сша / шесть: 6 (3.0%)
    сша / шухер / экс / чистая прибыль / чистая: 4 (2.0%)
    иране / войны иране / войны / песков назвал / мира: 4 (2.0%)
  [entities] top persons: {}
  [style]  avg_title=9.3 words, avg_lead=0.0 words

[analyze] lenta: 2883 records
  [noise] lenta: daily_mean=31.7, cv=0.436, stable=True
  [topics] top-5:
    2026спорт / февраля / 

In [18]:
# ── Топ-10 тем по каждому СМИ ────────────────────────────────────
import pandas as pd

TOPIC_OUTLET = "interfax"   # поменяйте на нужный slug

if TOPIC_OUTLET in analysis and analysis[TOPIC_OUTLET]:
    freq = analysis[TOPIC_OUTLET]["topic_freq"]
    print(f"\nТоп-10 тем ({config.OUTLETS[TOPIC_OUTLET]['name']}, последние {config.TOPIC_WINDOW} дней):\n")
    display(freq.head(10)[["cluster_name", "count", "pct"]])
else:
    print(f"Нет данных для {TOPIC_OUTLET} — сначала запустите Scrape+ETL")


Топ-10 тем (Интерфакс, последние 14 дней):



,cluster_name,count,pct
1,трамп / россии / ирана / млрд / ограничения,410,50.8
15,сша / ирана / президент / ираном / трамп,81,10.0
6,рф / мид рф / мид / рф январе / цб,67,8.3
0,иран / сша / проливе / ормузском проливе / орм...,37,4.6
17,области / путин / запорожской / запорожской об...,32,4.0
9,глава / глава мид / мид / сша / мид ирана,21,2.6
16,москве / число / летевших / подлете / сбитых,18,2.2
5,аэропорт / работу / приостановил / возобновил ...,17,2.1
14,безопасности / москвы / предложил / центре мос...,17,2.1
10,ес / ек / рф / санкции / стран ес,16,2.0


In [19]:
# ── Noise check: стабильность объёма по всем СМИ ─────────────────
noise_rows = []
for slug, res in analysis.items():
    if not res:
        continue
    n = res["noise"]
    noise_rows.append({
        "outlet":       slug,
        "daily_mean":   n.get("daily_mean"),
        "cv":           n.get("cv"),
        "stable":       "✅" if n.get("stable") else "⚠️",
        "top_rubric":   n.get("top_rubric"),
        "top_share_%":  round(n.get("top_rubric_share", 0) * 100, 1),
        "rubric_noisy": "⚠️" if n.get("rubric_noisy") else "✅",
    })

pd.DataFrame(noise_rows)

,outlet,daily_mean,cv,stable,top_rubric,top_share_%,rubric_noisy
0,kommersant,83.2,0.928,⚠️,Мир,36.9,✅
1,vedomosti,66.0,0.265,✅,Политика / Международные новости,37.9,✅
2,lenta,31.7,0.436,✅,,93.1,⚠️
3,interfax,59.0,0.086,✅,,99.5,⚠️


In [20]:
# ── Топ сущностей (персоны, организации) ─────────────────────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ENT_OUTLET = "kommersant"   # поменяйте
ENT_TYPE   = "persons"  # persons | orgs | locations

if ENT_OUTLET in analysis and analysis[ENT_OUTLET]:
    ents = analysis[ENT_OUTLET]["entities"].get(ENT_TYPE, pd.Series())
    if not ents.empty:
        fig = go.Figure(go.Bar(
            x=ents.values[:20][::-1],
            y=ents.index[:20][::-1],
            orientation="h"
        ))
        fig.update_layout(
            title=f"Топ упоминаний: {ENT_TYPE} — {config.OUTLETS[ENT_OUTLET]['name']}",
            height=500, margin=dict(l=200)
        )
        fig.show()
    else:
        print(f"Нет сущностей типа '{ENT_TYPE}' для {ENT_OUTLET}")

In [1]:
# ── Динамика публикаций по дням (все СМИ) ────────────────────────
import plotly.express as px
from etl import load_all_clean

df_all = load_all_clean()
if not df_all.empty:
    daily = (
        df_all.groupby([df_all["published_at"].dt.date, "outlet"])
              .size()
              .reset_index(name="count")
    )
    daily.columns = ["date", "outlet", "count"]
    fig = px.line(daily, x="date", y="count", color="outlet",
                  title="Количество публикаций по дням",
                  labels={"count": "Публикаций", "date": "Дата"})
    fig.show()
else:
    print("Нет данных — запустите Scrape + ETL")

---
## Ячейка 4 — Backtest: holdout-проверка

> Обучение на `[all data − 7 days]`, тест на последних 7 днях.
> Методы: **inertia**, **frequency**, **calendar**.

In [3]:
import config
from backtester import backtest_all

# ── Настройки ────────────────────────────────────────────────────
BT_OUTLETS     = config.OUTLET_SLUGS
BT_HOLDOUT     = config.BACKTEST_DAYS      # 7 дней
BT_METHODS     = ["inertia", "frequency", "calendar"]
# ────────────────────────────────────────────────────────────────

bt_results = backtest_all(
    slugs=BT_OUTLETS,
    holdout_days=BT_HOLDOUT,
    methods=BT_METHODS,
)

print("\n── Бэктест завершён ──")
for slug, days in bt_results.items():
    print(f"  {slug:<12} {len(days)} дней")


  [backtest] No clean data for kommersant
  [backtest] saved → c:\Users\xaxhd\OneDrive\Рабочий стол\TEst\titles\data\forecasts\backtest_kommersant_20260329.json

[backtest] kommersant: holdout 2026-03-23 → 2026-03-29
  2026-03-27: 190 actual articles
  2026-03-28: 81 actual articles
  2026-03-29: 54 actual articles
  [backtest] saved → c:\Users\xaxhd\OneDrive\Рабочий стол\TEst\titles\data\forecasts\backtest_kommersant_20260329.json

[backtest] deleted: holdout 2026-03-23 → 2026-03-29
  2026-03-28: 67 actual articles
  2026-03-29: 48 actual articles
  [backtest] saved → c:\Users\xaxhd\OneDrive\Рабочий стол\TEst\titles\data\forecasts\backtest_deleted_20260329.json

[backtest] lenta: holdout 2026-03-23 → 2026-03-29
  2026-03-23: 30 actual articles
  2026-03-24: 29 actual articles
  2026-03-25: 30 actual articles
  2026-03-26: 29 actual articles
  2026-03-27: 28 actual articles
  2026-03-28: 76 actual articles
  2026-03-29: 152 actual articles
  [backtest] saved → c:\Users\xaxhd\OneDrive\Р

In [8]:
# Просмотр одного дня бэктеста
import json

BT_OUTLET = 
BT_DAY_IDX = 0    # индекс дня в holdout (0 = самый ранний)

if BT_OUTLET in bt_results and bt_results[BT_OUTLET]:
    day = bt_results[BT_OUTLET][BT_DAY_IDX]
    print(f"Дата: {day['date']}")
    print(f"\nФактические заголовки ({len(day['actual']['titles'])})")
    for t in day["actual"]["titles"][:10]:
        print(f"  ► {t}")
    print(f"\nПрогноз 'frequency' (топ-5 тем)")
    for p in day["predictions"].get("frequency", [])[:5]:
        print(f"  • {p.get('topic_label', '')[:80]}")
else:
    print(f"Нет данных бэктеста для {BT_OUTLET} — запустите ячейку выше")

Дата: 2026-03-28

Фактические заголовки (67)
  ► За ночь над Россией уничтожили более 150 БПЛА
  ► Axios: Рубио и Каллас спорили о России на встрече G7
  ► В аэропортах Казани и Нижнекамска сняли ограничения на полеты
  ► Россия за 10 лет нарастила выпуск устриц и гребешков в 10 раз
  ► Володин за год направил на благотворительность 61,4 млн рублей
  ► Дмитриев: диалог России и США продолжится, несмотря на противников
  ► ФСБ предотвратила организованный Киевом теракт в Ставрополе
  ► WSJ: дефицит ракет ПВО у США грозит оставить Украину без средств защиты
  ► Руденко: Москва примет ответные меры против Сеула в случае отправки оружия Киеву
  ► Запуск лунной миссии может быть отложен из-за выброса плазмы на Солнце

Прогноз 'frequency' (топ-5 тем)
  • россии / могут / назначен / иран / аракчи
  • рф / сша / чистая / чистая прибыль / украины
  • сша / второй / чистая / участии / украины
  • путин / назвал / россии / чистая / участии
  • россия / нужна / украины / чистая / трампа


---
## Ячейка 5 — Metrics: оценка качества

| Метрика | Целевое значение | Описание |
|---------|-----------------|----------|
| topic_hit_rate | ≥ 0.30 | Совпадение тематики (Jaccard) |
| entity_match_f1 | ≥ 0.20 | Совпадение персон и орг (F1) |
| semantic_similarity | ≥ 0.45 | Семантическая близость (cosine) |
| style_match | ≥ 0.30 | Соответствие стилю СМИ |
| diversity_score | ≥ 0.70 | Отсутствие дублей среди прогнозов |

In [9]:
import config
from metrics import evaluate_all

metrics_reports = evaluate_all(
    slugs=config.OUTLET_SLUGS,
    methods=["inertia", "frequency", "calendar"]
)


  [metrics] No backtest data for kommersant
  [metrics] sentence-transformers not available: No module named 'sentence_transformers'
  [metrics] sentence-transformers not available: No module named 'sentence_transformers'
  [metrics] sentence-transformers not available: No module named 'sentence_transformers'

[metrics] kommersant | method=frequency
  topic_hit_rate            0.0000
  entity_match_f1           0.0000
  semantic_similarity       0.0000
  style_match               0.0000
  diversity_score           1.0000
  [metrics] sentence-transformers not available: No module named 'sentence_transformers'
  [metrics] sentence-transformers not available: No module named 'sentence_transformers'

[metrics] deleted | method=frequency
  topic_hit_rate            0.0000
  entity_match_f1           0.0000
  semantic_similarity       0.0000
  style_match               0.0000
  diversity_score           1.0000
  [metrics] sentence-transformers not available: No module named 'sentence_transfo

In [10]:
# ── Сводная таблица метрик ────────────────────────────────────────
import pandas as pd

TARGETS = {
    "topic_hit_rate":      0.30,
    "entity_match_f1":     0.20,
    "semantic_similarity": 0.45,
    "style_match":         0.30,
    "diversity_score":     0.70,
}

rows = []
for slug, rep in metrics_reports.items():
    row = {"outlet": slug, "method": rep.get("method")}
    for metric, target in TARGETS.items():
        val = rep.get(metric, 0)
        row[metric] = f"{val:.3f}  {'✅' if val >= target else '❌'}"
    rows.append(row)

pd.set_option("display.max_colwidth", 20)
pd.DataFrame(rows).set_index("outlet")

,method,topic_hit_rate,entity_match_f1,semantic_similarity,style_match,diversity_score
outlet,,,,,,
kommersant,frequency,0.000 ❌,0.000 ❌,0.000 ❌,0.000 ❌,1.000 ✅
vedomosti,frequency,0.000 ❌,0.000 ❌,0.000 ❌,0.000 ❌,1.000 ✅
lenta,frequency,0.095 ❌,0.000 ❌,0.000 ❌,0.000 ❌,1.000 ✅
interfax,frequency,0.000 ❌,0.000 ❌,0.000 ❌,0.000 ❌,1.000 ✅


In [11]:
# ── Визуализация метрик (radar chart) ────────────────────────────
import plotly.graph_objects as go

metric_cols = list(TARGETS.keys())
fig = go.Figure()

for slug, rep in metrics_reports.items():
    values = [rep.get(m, 0) for m in metric_cols]
    values_closed = values + [values[0]]  # close polygon
    cats_closed   = metric_cols + [metric_cols[0]]
    fig.add_trace(go.Scatterpolar(
        r=values_closed,
        theta=cats_closed,
        fill="toself",
        name=config.OUTLETS[slug]["name"],
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title="Метрики качества прогноза по СМИ",
    showlegend=True,
)
fig.show()

---
## Ячейка 6 — Forecast: прогноз на 02.04.2026

> Если OpenRouter недоступна — установите `USE_LLM = False`, пайплайн сгенерирует baseline-прогнозы без LLM.
>
> Итоговый JSON сохраняется в `data/forecasts/forecast_2026-04-02.json`

In [12]:
import config, datetime
from forecaster import forecast_all

# ── Настройки ────────────────────────────────────────────────────
TARGET_DATE    = datetime.date(2026, 4, 2)
USE_LLM        = True   # False — пропустить генерацию заголовков через Ollama
FORECAST_SLUGS = config.OUTLET_SLUGS
# ────────────────────────────────────────────────────────────────

forecast_reports = forecast_all(
    slugs=FORECAST_SLUGS,
    target_date=TARGET_DATE,
    use_llm=USE_LLM,
)


  [forecast] No clean data for kommersant

[forecast] Коммерсантъ → 2026-04-02
  [llm] kommersant | topic: россии / делу / пресс служба / служба / пресс...
  [llm] kommersant | topic: сша / ирана / агентство / reuters / трампа...
  [llm] kommersant | topic: года / марта / вице / премьер / россии...

[forecast] Ведомости → 2026-04-02
  [llm] deleted | topic: сша / иран / ираном / апреля / ирана...
  [llm] deleted | topic: путин / россии / россия / президентом / москва...
  [llm] deleted | topic: рф / ушаков / депутатами / сша / шесть...

[forecast] Лента.ру → 2026-04-02
  [llm] lenta | topic: 2026спорт / февраля / 2026забота / февраля 2026спорт / 00...
  [llm] lenta | topic: марта 2026мир / марта / 2026мир / трамп / сша...
  [llm] lenta | topic: марта 2026бывший / стало известно / известно / стало / 2026б...

[forecast] Интерфакс → 2026-04-02
  [llm] interfax | topic: трамп / россии / ирана / млрд / ограничения...
  [llm] interfax | topic: сша / ирана / президент / ираном / трамп...
  [

In [13]:
# ── Сводка по каждому СМИ ────────────────────────────────────────
for slug, rep in forecast_reports.items():
    if not rep:
        continue
    name   = rep.get("outlet_name", slug)
    preds  = rep.get("predictions", [])
    llm    = [p for p in preds if p.get("method") == "llm"]
    cal    = [p for p in preds if p.get("method") == "calendar"]
    freq   = [p for p in preds if p.get("method") == "frequency"]
    iner   = [p for p in preds if p.get("method") == "inertia"]

    print(f"\n{'='*60}")
    print(f"  {name} ({slug})")
    print(f"{'='*60}")
    print(f"  Топ-темы: {', '.join(rep.get('top_topics', [])[:3])}")
    print(f"  Контекст событий:\n    {rep.get('events_context', '').replace(chr(10), chr(10)+'    ')}")
    print(f"\n  Baseline — Инерция ({len(iner)} тем):")
    for p in iner[:5]:
        print(f"    · {p.get('topic_label', '')[:70]}")
    print(f"\n  Baseline — Частотность ({len(freq)} тем):")
    for p in freq[:5]:
        print(f"    · {p.get('topic_label', '')[:70]}")
    print(f"\n  Календарь событий ({len(cal)}):")
    for p in cal:
        print(f"    • {p.get('title', '')}")
    if llm:
        print(f"\n  LLM-заголовки ({len(llm)}):")
        for p in llm[:8]:
            print(f"    ▶ {p.get('title', '')}")
            lead = p.get("lead")
            if lead:
                print(f"      {lead[:120]}")


  Коммерсантъ (kommersant)
  Топ-темы: россии / делу / пресс служба / служба / пресс, сша / ирана / агентство / reuters / трампа, года / марта / вице / премьер / россии
  Контекст событий:
    - 01.04.2026: [holiday] День смеха (1 апреля)
    - 02.04.2026: [economics] Ожидается публикация PMI-индексов промышленного производства (мировые)
    - 02.04.2026: [politics] Заседание Госдумы (плановое)
    - 03.04.2026: [economics] Заседание Банка России / пресс-конференция (ближайшее)

  Baseline — Инерция (0 тем):

  Baseline — Частотность (10 тем):
    · россии / делу / пресс служба / служба / пресс
    · сша / ирана / агентство / reuters / трампа
    · года / марта / вице / премьер / россии
    · области / всу / telegram / канале / telegram канале
    · мид / мира / сша / которых / связи

  Календарь событий (4):
    • [Шаблон] Коммерсантъ: День смеха (1 апреля)
    • [Шаблон] Коммерсантъ: Ожидается публикация PMI-индексов промышленного производства (мировые)
    • [Шаблон] Коммерсантъ: З

---
## Ячейка 7 — Results: просмотр итогового JSON

In [14]:
import json, os
from IPython.display import JSON

json_path = os.path.join(config.FORECASTS_DIR, "forecast_2026-04-02.json")

if os.path.exists(json_path):
    with open(json_path, encoding="utf-8") as f:
        forecast_json = json.load(f)
    print(f"Файл: {json_path}")
    print(f"СМИ в файле: {list(forecast_json.keys())}")
    # Интерактивный JSON-просмотр в Jupyter
    JSON(forecast_json)
else:
    print(f"Файл не найден: {json_path}")
    print("Сначала запустите ячейку Forecast (шаг 6).")

Файл: c:\Users\xaxhd\OneDrive\Рабочий стол\TEst\titles\data\forecasts\forecast_2026-04-02.json
СМИ в файле: ['kommersant', 'kommersant', 'deleted', 'lenta', 'interfax']


In [15]:
# ── Таблица всех LLM-прогнозов ────────────────────────────────────
import pandas as pd

rows = []
for slug, rep in forecast_reports.items():
    if not rep:
        continue
    for p in rep.get("predictions", []):
        if p.get("method") == "llm":
            rows.append({
                "СМИ":     config.OUTLETS[slug]["name"],
                "Рубрика": p.get("rubric", ""),
                "Заголовок": p.get("title", ""),
                "Лид":     (p.get("lead") or "")[:120],
            })

if rows:
    df_llm = pd.DataFrame(rows)
    pd.set_option("display.max_colwidth", 80)
    display(df_llm)
else:
    print("LLM-прогнозы не сгенерированы. Проверьте Ollama или запустите с USE_LLM=True.")

,СМИ,Рубрика,Заголовок,Лид
0,Коммерсантъ,россии,** «Рост инвестиций в промышленность замедлился до двухлетнего минимума»,В преддверии публикации PMI-индексов промышленного производства стало извест...
1,Коммерсантъ,сша,Я расскажу о 5 возможных новостях.,
2,Коммерсантъ,сша,** США и Иран на пороге конфликта,"** Обстановка в регионе продолжает оставаться напряженной, с обеих сторон сл..."
3,Коммерсантъ,сша,** Экономика США готовится к неизбежному,"** Директор Агентства по статистике заявил, что новое осложнение в отношения..."
4,Коммерсантъ,сша,** СМИ: США рассматривают военные действия против Ирана,"** Reuters сообщает, что президент Трамп в ближайшее время должен обсудить в..."
5,Коммерсантъ,сша,** Кризис на Ближнем Востоке: реакция мирового бизнеса,** Многие компании уже выражают обеспокоенность потенциальными последствиями...
6,Коммерсантъ,сша,** В США ждут решения по Ирану,"** Представители министерства обороны подтвердили, что президент Трамп продо..."
7,Коммерсантъ,года,Высокие ставки на фоне роста инфляции,В марте текущего года средневзвешенная процентная ставка по банковским вклад...
8,Коммерсантъ,года,Рост промышленного производства ожидается в мире и России,В понедельник будут опубликованы мировые PMI-индексы промышленного производс...
9,Коммерсантъ,года,Обсуждение бюджета пройдет в Госдуме без опозданий,"В понедельник состоится плановое заседание Государственной Думы, на котором ..."


In [18]:
# ── Экспорт прогнозов в Excel ─────────────────────────────────────
import config, datetime, os
import pandas as pd

TARGET_DATE = datetime.date(2026, 4, 2)
EXPORT_PATH = os.path.join(config.FORECASTS_DIR, f"forecast_{TARGET_DATE}.xlsx")

rows = []
for slug, rep in forecast_reports.items():
    if not rep:
        continue
    outlet_name = config.OUTLETS[slug]["name"]
    for p in rep.get("predictions", []):
        rows.append({
            "СМИ":         outlet_name,
            "Метод":       p.get("method", ""),
            "Рубрика":     p.get("rubric", "") or p.get("topic_label", ""),
            "Заголовок":   p.get("title", "") or p.get("topic_label", ""),
            "Лид":         (p.get("lead") or ""),
            "Уверенность": p.get("score", ""),
        })

if rows:
    df_export = pd.DataFrame(rows)

    with pd.ExcelWriter(EXPORT_PATH, engine="openpyxl") as writer:
        # Лист 1: все прогнозы
        df_export.to_excel(writer, sheet_name="Все прогнозы", index=False)

        # Лист 2: только LLM-заголовки
        df_llm = df_export[df_export["Метод"] == "llm"]
        if not df_llm.empty:
            df_llm.to_excel(writer, sheet_name="LLM заголовки", index=False)

        # Лист 3+: по одному листу на СМИ
        for slug, rep in forecast_reports.items():
            if not rep:
                continue
            name = config.OUTLETS[slug]["name"]
            df_outlet = df_export[df_export["СМИ"] == name]
            df_outlet.to_excel(writer, sheet_name=slug[:31], index=False)

    print(f"✅ Файл сохранён: {EXPORT_PATH}")
    print(f"   Строк: {len(df_export)}  |  Листов: {2 + len(forecast_reports)}")
else:
    print("⚠️  Нет данных для экспорта — сначала запустите ячейку Forecast.")


✅ Файл сохранён: c:\Users\xaxhd\OneDrive\Рабочий стол\TEst\titles\data\forecasts\forecast_2026-04-02.xlsx
   Строк: 110  |  Листов: 7


In [16]:
# ── Таблица календарных событий на дату ──────────────────────────
from event_calendar import load_events

events = load_events(target_date=datetime.date(2026, 4, 2), window_days=3)
pd.DataFrame(events)[["date", "event_type", "description", "outlets"]]

,date,event_type,description,outlets
0,2026-04-01,holiday,День смеха (1 апреля),"[rbc, kommersant, lenta, interfax, vedomosti]"
1,2026-04-02,economics,Ожидается публикация PMI-индексов промышленного производства (мировые),"[rbc, vedomosti, kommersant, interfax]"
2,2026-04-02,politics,Заседание Госдумы (плановое),"[rbc, interfax, kommersant]"
3,2026-04-02,economics,Статистика рынка труда США (еженедельная),"[rbc, vedomosti]"
4,2026-04-03,economics,Заседание Банка России / пресс-конференция (ближайшее),"[rbc, vedomosti, kommersant, interfax]"
5,2026-04-04,sport,Тур де Франс / футбольные матчи РПЛ (тур),"[lenta, rbc]"


---
## Быстрый запуск всего пайплайна одной ячейкой

In [19]:
# !! Выполнит весь пайплайн последовательно !!
# Раскомментируйте нужный вариант
import subprocess, sys

# ВАРИАНТ 1: только прогноз без LLM (быстро, если данные уже собраны)
# subprocess.run([sys.executable, "main.py", "--mode", "forecast",
#                 "--target", "2026-04-02", "--no-llm"], check=True)

# ВАРИАНТ 2: полный пайплайн с LLM (~20-40 мин)
#subprocess.run([sys.executable, "main.py", "--mode", "all",
#                  "--target", "2026-04-02"], check=True)

# ВАРИАНТ 3: только для одного СМИ
# subprocess.run([sys.executable, "main.py", "--mode", "all",
#                 "--outlets", "kommersant", "--target", "2026-04-02"], check=True)

print("Раскомментируйте нужный вариант выше и запустите ячейку.")


Раскомментируйте нужный вариант выше и запустите ячейку.
